# This gave very poor results



In [ ]:
# ── Cell 1: Install dependencies ────────────────────────────────
!pip install -q "transformers>=4.45.0" "datasets>=2.14" "peft>=0.7" "accelerate>=0.25" \
    "bitsandbytes" "jiwer" "evaluate" "torchaudio" "librosa" "huggingface_hub"
print('✅ Installed')

In [ ]:
# ── Cell 2: Imports & Config ────────────────────────────────────
import os, gc, re, json, glob
import torch
import numpy as np
import unicodedata
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from urllib.parse import urlparse, parse_qs

import torchaudio
import librosa
from datasets import Dataset
from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    WhisperTokenizer,
    WhisperFeatureExtractor,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    TrainerCallback,
)
from peft import LoraConfig, get_peft_model, PeftModel
import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        total_gb = props.total_memory / 1e9 if hasattr(props, 'total_memory') else getattr(props, 'total_mem', 0) / 1e9
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} — {total_gb:.1f} GB")

# ═══════════════════════════════════════════════════════════════
# CONFIGURATION — Change these values
# ═══════════════════════════════════════════════════════════════
BASE_MODEL = "bengaliAI/tugstugi_bengaliai-asr_whisper-medium"
LANGUAGE = "bn"
TASK = "transcribe"

# ⚠ CHANGE THESE to your HuggingFace repo names
HF_OUTPUT_REPO = "lucius-40/whisper-medium-bengali-finetuned-tustugi"

# External dataset paths on Kaggle
EXT_AUDIO_DIR = "/kaggle/input/banglalong-asr/asr dataset/audio"
EXT_TRANSCRIPTION_DIR = "/kaggle/input/banglalong-asr/asr dataset/transcription"

# Phase 1: first 125 files. Set to None for all files.
PHASE1_FILE_COUNT = 125

# ── LoRA ──
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"]

# ── Training ──
LEARNING_RATE = 5e-5         # Conservative for large dataset
WARMUP_STEPS = 300
MAX_STEPS = 4500             # ~3.5 epochs over ~39k segments
EVAL_STEPS = 450             # Eval roughly once per epoch
SAVE_STEPS = 450
PER_DEVICE_BATCH = 4
GRAD_ACCUM = 4               # Effective batch = 4 * 4 * n_gpu = 32

# ── Segment filtering ──
MIN_DURATION_S = 2.0
MAX_DURATION_S = 25.0
MIN_TEXT_CHARS = 5
MIN_CHARS_PER_SEC = 1.0      # Bengali speech lower bound
MAX_CHARS_PER_SEC = 20.0     # Bengali speech upper bound

# ── HF push interval ──
# Push to HuggingFace every N training steps (~10 files worth)
# ~39k segments / 32 eff_batch = 1219 steps/epoch. 10 files ≈ 100 steps.
HF_PUSH_EVERY_STEPS = 450    # Aligned with save_steps

OUTPUT_DIR = "/kaggle/working/whisper-bengali-lora"

print(f"✅ Config ready — {BASE_MODEL}")
print(f"   LoRA targets: {LORA_TARGETS}")
print(f"   Effective batch: {PER_DEVICE_BATCH} × {GRAD_ACCUM} × {max(1, torch.cuda.device_count())} = {PER_DEVICE_BATCH * GRAD_ACCUM * max(1, torch.cuda.device_count())}")

In [ ]:
# ── Cell 3: HuggingFace Login ───────────────────────────────────
from huggingface_hub import login
import os

hf_token = os.environ.get('hugging_face', None)
if hf_token:
    login(token=hf_token)
    print('✅ Logged in via HF_TOKEN secret')
else:
    login()  # Will prompt for token

In [ ]:
# ── Cell 4: Build Dataset from External JSON + WAV ──────────────
# Maps each JSON's video_url → YouTube video ID → matches to WAV filename.
# Resolves overlapping segments by keeping the longer one.
# Streams through audio files one at a time (no chunked files saved to disk).

# ── Helper: extract YouTube video ID from URL ──
def extract_video_id(url):
    """Extract YouTube video ID from various URL formats."""
    if not url:
        return None
    # Handle standard youtube.com/watch?v=ID
    parsed = urlparse(url)
    if 'youtube.com' in parsed.hostname or 'www.youtube.com' in parsed.hostname:
        qs = parse_qs(parsed.query)
        return qs.get('v', [None])[0]
    # Handle youtu.be/ID
    if 'youtu.be' in parsed.hostname:
        return parsed.path.lstrip('/')
    return None

# ── Helper: resolve overlapping segments ──
def resolve_overlaps(segments):
    """
    Given a list of {text, start, end} segments sorted by start time,
    resolve overlaps by keeping the LONGER segment when two overlap.
    Short filler words (হ্যাঁ, না, etc.) that cause overlaps are dropped.
    """
    if not segments:
        return []
    
    # Sort by start time, break ties by longer segment first
    sorted_segs = sorted(segments, key=lambda s: (s['start'], -(s['end'] - s['start'])))
    
    resolved = [sorted_segs[0]]
    for seg in sorted_segs[1:]:
        prev = resolved[-1]
        # Check overlap: current segment starts before previous ends
        if seg['start'] < prev['end']:
            # Keep the longer one
            prev_dur = prev['end'] - prev['start']
            seg_dur = seg['end'] - seg['start']
            if seg_dur > prev_dur:
                # Replace previous with current (current is longer)
                resolved[-1] = seg
            # else: keep previous (it's longer or equal), skip current
        else:
            # No overlap — keep both
            resolved.append(seg)
    
    return resolved

# ── Helper: clean text (same normalization as inference pipeline) ──
def clean_text(text):
    """NFC normalize, strip punctuation, collapse whitespace."""
    if not text:
        return ""
    text = unicodedata.normalize('NFC', text.strip())
    # Remove Bengali and standard punctuation
    for p in '।.,;:!?\'"()[]{}—–-–―…॰':
        text = text.replace(p, '')
    # Remove sound effect annotations like [সশব্দ হাসি]
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ── Helper: load audio ──
def load_full_audio(wav_path, target_sr=16000):
    """Load audio file → mono float32 numpy @ target_sr."""
    try:
        audio, sr = torchaudio.load(wav_path)
        if audio.shape[0] > 1:
            audio = torch.mean(audio, dim=0, keepdim=True)
        if sr != target_sr:
            audio = torchaudio.transforms.Resample(sr, target_sr)(audio)
        return audio.squeeze().numpy(), target_sr
    except Exception:
        audio, sr = librosa.load(wav_path, sr=target_sr, mono=True)
        return audio, sr

# ── Step 1: Discover and map all JSON → WAV pairs ──
print("Scanning external dataset …")
print(f"  Audio dir:         {EXT_AUDIO_DIR}")
print(f"  Transcription dir: {EXT_TRANSCRIPTION_DIR}")

# Build lookup: video_id → audio_path
audio_files = {}
if os.path.exists(EXT_AUDIO_DIR):
    for f in os.listdir(EXT_AUDIO_DIR):
        if f.endswith('.wav'):
            vid_id = os.path.splitext(f)[0]  # filename IS the video ID
            audio_files[vid_id] = os.path.join(EXT_AUDIO_DIR, f)
    print(f"  Found {len(audio_files)} WAV files")
else:
    print(f"  ⚠ Audio directory not found!")

# Build list of matched (json_path, wav_path) pairs
json_files = sorted(glob.glob(os.path.join(EXT_TRANSCRIPTION_DIR, "*.json")))
print(f"  Found {len(json_files)} JSON files")

matched_pairs = []
unmatched = []
for jf in json_files:
    with open(jf, 'r', encoding='utf-8') as f:
        data = json.load(f)
    video_url = data.get('video_url', '')
    vid_id = extract_video_id(video_url)
    
    if vid_id and vid_id in audio_files:
        matched_pairs.append((jf, audio_files[vid_id], vid_id))
    else:
        # Try matching by audio_path field or json filename
        audio_path_field = data.get('audio_path', '')
        base = os.path.splitext(os.path.basename(audio_path_field))[0] if audio_path_field else None
        # Also try: json filename "0.json" → look for WAV by video_id
        if vid_id:
            unmatched.append((jf, vid_id))
        elif base:
            unmatched.append((jf, base))

print(f"\n  ✅ Matched {len(matched_pairs)} JSON→WAV pairs")
if unmatched:
    print(f"  ⚠ Unmatched: {len(unmatched)} (no WAV found for these video IDs)")
    for jf, vid in unmatched[:5]:
        print(f"     {os.path.basename(jf)} → {vid}")

# Apply phase limit
if PHASE1_FILE_COUNT and PHASE1_FILE_COUNT < len(matched_pairs):
    matched_pairs = matched_pairs[:PHASE1_FILE_COUNT]
    print(f"\n  Phase 1: Using first {PHASE1_FILE_COUNT} files")
else:
    print(f"\n  Using all {len(matched_pairs)} files")

# ── Step 2: Pre-scan to count segments and estimate dataset size ──
total_segments = 0
total_after_filter = 0
total_after_overlap = 0

for jf, wav_path, vid_id in matched_pairs:
    with open(jf, 'r', encoding='utf-8') as f:
        data = json.load(f)
    segments = data.get('segments', [])
    total_segments += len(segments)
    
    # Filter
    filtered = []
    for seg in segments:
        text = clean_text(seg.get('text', ''))
        start, end = seg.get('start', 0), seg.get('end', 0)
        dur = end - start
        
        if start >= end: continue
        if dur < MIN_DURATION_S or dur > MAX_DURATION_S: continue
        if len(text) < MIN_TEXT_CHARS: continue
        
        chars_per_sec = len(text) / dur
        if chars_per_sec < MIN_CHARS_PER_SEC or chars_per_sec > MAX_CHARS_PER_SEC: continue
        
        filtered.append({'text': text, 'start': start, 'end': end})
    
    total_after_filter += len(filtered)
    resolved = resolve_overlaps(filtered)
    total_after_overlap += len(resolved)

print(f"\n  Dataset statistics:")
print(f"    Raw segments:       {total_segments:,}")
print(f"    After filtering:    {total_after_filter:,}")
print(f"    After overlap res:  {total_after_overlap:,}")

est_steps_per_epoch = total_after_overlap // (PER_DEVICE_BATCH * GRAD_ACCUM * max(1, torch.cuda.device_count()))
print(f"    Est. steps/epoch:   {est_steps_per_epoch:,}")
print(f"    Est. epochs in {MAX_STEPS} steps: {MAX_STEPS / max(1, est_steps_per_epoch):.1f}")

In [ ]:
# ── Cell 5: Load Processor & Model ──────────────────────────────
print(f"Loading {BASE_MODEL} …")

feature_extractor = WhisperFeatureExtractor.from_pretrained(BASE_MODEL)
tokenizer = WhisperTokenizer.from_pretrained(BASE_MODEL, language=LANGUAGE, task=TASK)
processor = WhisperProcessor.from_pretrained(BASE_MODEL, language=LANGUAGE, task=TASK)

# Load model in fp16
model = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    use_cache=False,  # Required for gradient checkpointing
)

# Set language & task tokens for generation
model.generation_config.language = LANGUAGE
model.generation_config.task = TASK
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language=LANGUAGE, task=TASK
)

print(f"✅ Model loaded — {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M params")

In [ ]:
# ── Cell 6: Apply LoRA ──────────────────────────────────────────
# Target attention projections AND feed-forward layers for more capacity.
# This gives ~8M trainable params (1%) — significantly better adaptation
# for Bengali phonetics without approaching full-finetune cost.

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGETS,
    lora_dropout=LORA_DROPOUT,
    bias="none",
)

# Enable gradient checkpointing
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

# Apply LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# ── Cell 7: Build Arrow Dataset via Generator ───────────────────
# Each audio file is loaded ONCE, all segments sliced in memory,
# mel features extracted, text tokenized, then the audio is freed.
# HF datasets caches to Arrow format on disk — no OOM, no intermediate WAV files.

from tqdm import tqdm

def segment_generator():
    """
    Generator that yields preprocessed {input_features, labels} dicts.
    Loads each WAV once, processes all its segments, then frees memory.
    """
    for jf, wav_path, vid_id in tqdm(matched_pairs, desc="Processing audio files"):
        # Load JSON
        with open(jf, 'r', encoding='utf-8') as f:
            data = json.load(f)
        segments = data.get('segments', [])
        
        # Filter segments
        filtered = []
        for seg in segments:
            text = clean_text(seg.get('text', ''))
            start, end = seg.get('start', 0), seg.get('end', 0)
            dur = end - start
            
            if start >= end: continue
            if dur < MIN_DURATION_S or dur > MAX_DURATION_S: continue
            if len(text) < MIN_TEXT_CHARS: continue
            
            chars_per_sec = len(text) / dur
            if chars_per_sec < MIN_CHARS_PER_SEC or chars_per_sec > MAX_CHARS_PER_SEC: continue
            
            filtered.append({'text': text, 'start': start, 'end': end})
        
        # Resolve overlaps
        resolved = resolve_overlaps(filtered)
        
        if not resolved:
            continue
        
        # Load audio ONCE for this file
        try:
            full_audio, sr = load_full_audio(wav_path, target_sr=16000)
        except Exception as e:
            print(f"  ⚠ Failed to load {wav_path}: {e}")
            continue
        
        audio_len_samples = len(full_audio)
        
        for seg in resolved:
            start_sample = int(seg['start'] * sr)
            end_sample = int(seg['end'] * sr)
            
            # Bounds check
            start_sample = max(0, min(start_sample, audio_len_samples))
            end_sample = max(start_sample, min(end_sample, audio_len_samples))
            
            if end_sample - start_sample < int(MIN_DURATION_S * sr):
                continue
            
            audio_chunk = full_audio[start_sample:end_sample]
            
            # Extract mel features
            input_features = feature_extractor(
                audio_chunk,
                sampling_rate=sr,
                return_tensors="np",
            ).input_features[0]
            
            # Tokenize text
            labels = tokenizer(seg['text']).input_ids
            
            yield {
                "input_features": input_features,
                "labels": labels,
            }
        
        # Free memory for this audio file
        del full_audio
        gc.collect()

print("Building dataset from generator … (this processes all audio files)")
print("Arrow cache will be saved to disk — subsequent runs skip processing.\n")

full_ds = Dataset.from_generator(
    segment_generator,
    cache_dir="/kaggle/working/arrow_cache",
)

print(f"\n✅ Dataset built: {len(full_ds):,} segments")
print(f"   Features shape: {np.array(full_ds[0]['input_features']).shape}")
print(f"   Sample label length: {len(full_ds[0]['labels'])} tokens")

# Train/Val split — 90/10
ds_split = full_ds.train_test_split(test_size=0.1, seed=42)
train_ds = ds_split['train']
eval_ds = ds_split['test']

print(f"   Train: {len(train_ds):,} segments")
print(f"   Val:   {len(eval_ds):,} segments")

In [ ]:
# ── Cell 8: Data Collator ───────────────────────────────────────
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # Replace pad token id with -100 so it's ignored by loss
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        # Strip leading BOS if present in all sequences
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)
print("✅ Data collator ready")

In [ ]:
# ── Cell 9: WER Metric (with matching text cleaning) ────────────
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Replace -100 with pad_token_id for decoding
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    # Apply the SAME text cleaning used during training label preprocessing
    # This ensures eval WER matches training normalization exactly.
    def _clean(s):
        s = unicodedata.normalize('NFC', s.strip())
        for p in '।.,;:!?\'"()[]{}—–-–―…॰':
            s = s.replace(p, '')
        s = re.sub(r'\[.*?\]', '', s)
        s = re.sub(r'\s+', ' ', s).strip()
        return s
    
    pred_str = [_clean(s) for s in pred_str]
    label_str = [_clean(s) for s in label_str]

    # Filter out empty references (would cause div-by-zero in WER)
    valid = [(p, r) for p, r in zip(pred_str, label_str) if r]
    if not valid:
        return {"wer": 1.0}
    pred_str, label_str = zip(*valid)

    wer = wer_metric.compute(predictions=list(pred_str), references=list(label_str))
    return {"wer": wer}

print("✅ WER metric ready (with matched text cleaning)")

In [ ]:
# ── Cell 10: Training Arguments ─────────────────────────────────
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    
    # Batch & accumulation
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    
    # Learning rate
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    
    # Steps
    max_steps=MAX_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    
    # Precision & performance
    fp16=True,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    
    # Generation during eval
    predict_with_generate=True,
    generation_max_length=448,
    
    # Logging
    logging_steps=25,
    report_to="none",
    
    # Misc
    remove_unused_columns=False,
    label_names=["labels"],
    push_to_hub=False,
)

print("✅ Training arguments configured")
n_gpu = max(1, torch.cuda.device_count())
eff_batch = PER_DEVICE_BATCH * GRAD_ACCUM * n_gpu
print(f"   Effective batch size: {PER_DEVICE_BATCH} × {GRAD_ACCUM} × {n_gpu} = {eff_batch}")
print(f"   Max steps: {MAX_STEPS}")
print(f"   LR: {LEARNING_RATE}, Warmup: {WARMUP_STEPS}")
print(f"   Eval/Save every {EVAL_STEPS} steps")

In [ ]:
# ── Cell 11: Create Trainer & Train ─────────────────────────────

# ── Callback: Push merged model to HuggingFace at each save step ──
class PushToHubCallback(TrainerCallback):
    """
    At every save checkpoint, merge LoRA → push merged model to HF Hub.
    This gives you an updated model on HuggingFace every ~450 steps.
    """
    def __init__(self, processor, hf_repo, language, task):
        self.processor = processor
        self.hf_repo = hf_repo
        self.language = language
        self.task = task
    
    def on_save(self, args, state, control, model=None, **kwargs):
        if model is None:
            return
        step = state.global_step
        print(f"\n  📤 Pushing checkpoint at step {step} to {self.hf_repo} …")
        try:
            # Save adapter checkpoint, then merge + push a snapshot
            adapter_dir = os.path.join(args.output_dir, f"adapter-step-{step}")
            model.save_pretrained(adapter_dir)
            
            # Load base model fresh, merge, push
            base_for_merge = WhisperForConditionalGeneration.from_pretrained(
                BASE_MODEL,
                torch_dtype=torch.float16,
            )
            merged = PeftModel.from_pretrained(base_for_merge, adapter_dir)
            merged = merged.merge_and_unload()
            
            # Set generation config on merged model
            merged.generation_config.language = self.language
            merged.generation_config.task = self.task
            merged.generation_config.forced_decoder_ids = self.processor.get_decoder_prompt_ids(
                language=self.language, task=self.task
            )
            
            merged.push_to_hub(self.hf_repo, commit_message=f"Step {step}", private=True)
            self.processor.push_to_hub(self.hf_repo, commit_message=f"Step {step}", private=True)
            
            print(f"  ✅ Pushed merged model at step {step}")
            
            # Cleanup
            del base_for_merge, merged
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                
        except Exception as e:
            print(f"  ⚠ Push failed at step {step}: {e}")
            print(f"     (Training continues — adapter saved locally at {adapter_dir})")

push_callback = PushToHubCallback(processor, HF_OUTPUT_REPO, LANGUAGE, TASK)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
    callbacks=[push_callback],
)

model.config.use_cache = False

# Evaluate BEFORE training to get baseline WER on the external val set
print("Evaluating baseline (before fine-tuning) …")
baseline = trainer.evaluate()
print(f"Baseline WER: {baseline['eval_wer']:.4f}")

# Train!
print(f"\n{'='*60}")
print(f"STARTING LORA FINE-TUNING — Phase 1")
print(f"  {len(train_ds):,} train / {len(eval_ds):,} val segments")
print(f"  {MAX_STEPS} steps, LR={LEARNING_RATE}, LoRA r={LORA_R}")
print(f"  Model will be pushed to {HF_OUTPUT_REPO} every {SAVE_STEPS} steps")
print(f"{'='*60}\n")

train_result = trainer.train()

# Print results
print(f"\n{'='*60}")
print(f"TRAINING COMPLETE")
print(f"{'='*60}")
print(f"Train loss:    {train_result.training_loss:.4f}")
print(f"Train runtime: {train_result.metrics['train_runtime']:.0f}s ({train_result.metrics['train_runtime']/3600:.1f}h)")
print(f"Train samples/s: {train_result.metrics['train_samples_per_second']:.1f}")

# Final evaluation
print("\nFinal evaluation …")
final = trainer.evaluate()
print(f"Final WER: {final['eval_wer']:.4f}")
print(f"WER improvement: {baseline['eval_wer']:.4f} → {final['eval_wer']:.4f} ({(baseline['eval_wer'] - final['eval_wer'])*100:.1f}pp)")

In [ ]:
# ── Cell 12: Final Merge & Save ──────────────────────────────────
print("Saving final LoRA adapter …")
model.save_pretrained(f"{OUTPUT_DIR}/lora-adapter-final")
print(f"  Adapter saved to {OUTPUT_DIR}/lora-adapter-final")

print("\nMerging LoRA adapters into base model …")

# Load fresh base model and merge the best adapter
base_model_final = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
)

merged_model = PeftModel.from_pretrained(base_model_final, f"{OUTPUT_DIR}/lora-adapter-final")
merged_model = merged_model.merge_and_unload()

# Set generation config
merged_model.generation_config.language = LANGUAGE
merged_model.generation_config.task = TASK
merged_model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language=LANGUAGE, task=TASK
)

# Save locally
merged_model.save_pretrained(f"{OUTPUT_DIR}/merged-model")
processor.save_pretrained(f"{OUTPUT_DIR}/merged-model")
print(f"  Merged model saved to {OUTPUT_DIR}/merged-model")

print("✅ Model merge complete")

In [ ]:
# ── Cell 13: Final Upload to HuggingFace Hub ────────────────────
print(f"Uploading final merged model to {HF_OUTPUT_REPO} …")

merged_model.push_to_hub(HF_OUTPUT_REPO, commit_message="Final Phase 1 model", private=True)
processor.push_to_hub(HF_OUTPUT_REPO, commit_message="Final Phase 1 model", private=True)

print(f"✅ Model uploaded to https://huggingface.co/{HF_OUTPUT_REPO}")
print(f"\nTo use in your inference notebook:")
print(f'  ASR_MODEL = "{HF_OUTPUT_REPO}"')

# Cleanup GPU memory
del merged_model, base_model_final
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# ── Cell 14: Sanity Check — Decode a sample with finetuned model ─
from transformers import pipeline as hf_pipeline

print("Loading fine-tuned model for sanity check …")
test_pipe = hf_pipeline(
    task="automatic-speech-recognition",
    model=f"{OUTPUT_DIR}/merged-model",
    device=0 if torch.cuda.is_available() else -1,
    torch_dtype=torch.float16,
)

# Grab a val sample and decode
sample_idx = 0
sample = eval_ds[sample_idx]
# We need to recover audio from the mel features — instead, just compare label decoding
sample_labels = [l for l in sample['labels'] if l != -100]
gt_text = tokenizer.decode(sample_labels, skip_special_tokens=True)
print(f"\nGround truth (from labels): {gt_text[:200]}")

# To properly test, pick a random audio from the matched pairs
jf_test, wav_test, _ = matched_pairs[0]
with open(jf_test, 'r', encoding='utf-8') as f:
    test_data = json.load(f)
test_segs = test_data.get('segments', [])

# Find a clean segment
for seg in test_segs:
    dur = seg['end'] - seg['start']
    text = clean_text(seg.get('text', ''))
    if 3 <= dur <= 15 and len(text) >= 10:
        break

audio_full, sr = load_full_audio(wav_test)
chunk = audio_full[int(seg['start']*sr):int(seg['end']*sr)]

result = test_pipe(
    {"array": chunk, "sampling_rate": sr},
    generate_kwargs={"task": "transcribe", "language": "bn"},
)
pred = result.get("text", "").strip()

print(f"Ground truth:  {text}")
print(f"Prediction:    {pred}")

if text and pred:
    from jiwer import wer as compute_wer_fn
    sample_wer = compute_wer_fn(clean_text(text), clean_text(pred))
    print(f"Sample WER:    {sample_wer:.4f}")

print(f"\n✅ Sanity check done. Use in inference: ASR_MODEL = \"{HF_OUTPUT_REPO}\"")

# Phase 2 (Optional): Continue training on remaining 125 files

To train on the remaining 125 files in a **new Kaggle session**:

1. **Save Phase 1 adapter** as a Kaggle output dataset
2. In a new notebook, change these config values:
```python
PHASE1_FILE_COUNT = None          # Use ALL files
# OR skip first 125 and use next 125:
matched_pairs = matched_pairs[125:250]
```
3. Load the Phase 1 LoRA adapter instead of applying fresh LoRA:
```python
model = PeftModel.from_pretrained(model, "/kaggle/input/phase1-lora/lora-adapter-final")
```
4. Use lower LR: `LEARNING_RATE = 2e-5`
5. Use fewer steps: `MAX_STEPS = 3000`, `WARMUP_STEPS = 150`

The HF repo will be updated in-place, so the inference notebook always uses the latest model.